# Real-ESRGAN Official Repo — รันตรงๆ

In [ ]:
# @title 1. โคลน repo + ติดตั้ง
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install -r requirements.txt --quiet
!python setup.py develop --quiet
print("Done")

In [ ]:
# @title 2. Patch basicsr ให้ใช้กับ torchvision ใหม่ได้
# แก้ไขไฟล์ degradations.py โดยตรง
filepath = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

with open(filepath, "r") as f:
    content = f.read()

# แทนที่ import ที่ถูกลบ
content = content.replace(
    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
    "from torchvision.transforms.functional import rgb_to_grayscale"
)

with open(filepath, "w") as f:
    f.write(content)

# ทดสอบ import
from basicsr.archs.rrdbnet_arch import RRDBNet
print("Patch OK")

In [ ]:
# @title 3. อัปโหลดรูป
from google.colab import files
uploaded = files.upload()
input_path = list(uploaded.keys())[0] if uploaded else None
import shutil
if input_path:
    shutil.copy(input_path, "inputs/")
print(f"Input: inputs/{input_path}")

In [ ]:
# @title 4. รัน official inference (x2plus)
import time, os

!python inference_realesrgan.py \
    -i inputs/{input_path} \
    -o results \
    -n RealESRGAN_x2plus \
    -s 2 \
    --tile 400

import glob
result_files = sorted(glob.glob("results/*_out.*"))
if result_files:
    result_path = result_files[0]
    print(f"Result: {result_path}")

In [ ]:
# @title 5. แสดงผล + ดาวน์โหลด
from google.colab.patches import cv2_imshow
from google.colab import files
import cv2

if result_files:
    img = cv2.imread(result_files[0])
    cv2_imshow(img)
    files.download(result_files[0])